# MENG 417 - Introduction to Neural Networks

The homework focuses on:

- learning the weights of a linear neuron with gradient descent;
- understanding when one perceptron is not enough;
- constructing a two-perceptron classifier by hand;
- implementing gradient descent in Python;
- supervised vs unsupervised learning;
- hard-threshold versus raw linear outputs;
- sigmoid outputs and probability-like confidence.

## Source-derived data

### Question 1

$$
\mathbf{x}_1=
\begin{bmatrix}
2\\
5\\
3
\end{bmatrix},
\qquad
y_1=850
$$

$$
\mathbf{x}_2=
\begin{bmatrix}
1\\
2\\
2
\end{bmatrix},
\qquad
y_2=450
$$

$$
\mathbf{x}_3=
\begin{bmatrix}
1\\
2\\
1
\end{bmatrix},
\qquad
y_3=300.
$$

### Question 2

Blue points:

$$
(-1,-1),\qquad(1,1),\qquad(2,2)
$$

Red points:

$$
(0,-1),\qquad(-1,0),\qquad(2,1),\qquad(1,2).
$$

> All displayed mathematics uses `$$ ... $$`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Question 1 — Learn the Weights of the Linear Neuron with Gradient Descent

The professor asks us to determine the weights of a **linear neuron**.

Because each input vector has three components and the question asks for the **weights**, the natural linear-neuron model is

$$
\boxed{
\hat y
=
\mathbf{w}^T\mathbf{x}
=
w_1x_1+w_2x_2+w_3x_3
}
$$

with no separate bias term.

The three training examples can be placed in matrix form:

$$
X
=
\begin{bmatrix}
2&5&3\\
1&2&2\\
1&2&1
\end{bmatrix},
$$

$$
\mathbf{y}
=
\begin{bmatrix}
850\\
450\\
300
\end{bmatrix}.
$$

The predictions for all three examples are

$$
\hat{\mathbf y}=X\mathbf w.
$$

---

## Step 1 — Choose the loss function

A standard squared-error objective is

$$
J(\mathbf w)
=
\frac{1}{2N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2,
$$

where

$$
N=3.
$$

In matrix form,

$$
\boxed{
J(\mathbf w)
=
\frac{1}{2N}
\|X\mathbf w-\mathbf y\|_2^2.
}
$$

---

## Step 2 — Derive the gradient

Let

$$
\mathbf e
=
X\mathbf w-\mathbf y.
$$

Then

$$
J(\mathbf w)
=
\frac{1}{2N}
\mathbf e^T\mathbf e.
$$

The gradient is

$$
\boxed{
\nabla J(\mathbf w)
=
\frac{1}{N}
X^T(X\mathbf w-\mathbf y).
}
$$

---

## Step 3 — Gradient-descent update

Gradient descent updates the weights using

$$
\boxed{
\mathbf w^{(k+1)}
=
\mathbf w^{(k)}
-
\eta
\nabla J(\mathbf w^{(k)}).
}
$$

The homework does not prescribe an initialization or learning rate, so we state the choices explicitly:

$$
\boxed{
\mathbf w^{(0)}=
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix}
}
$$

and

$$
\boxed{\eta=0.05}.
$$

This learning rate is small enough for stable convergence for this dataset.

---

## Step 4 — The weights to which gradient descent converges

Running the iterations gives

$$
\boxed{
\mathbf w
\approx
\begin{bmatrix}
-50\\
100\\
150
\end{bmatrix}.
}
$$

The values are effectively exact because the three training equations are consistent.

---

## Step 5 — Verify the learned neuron

### First sample

$$
\hat y_1
=
(-50)(2)+(100)(5)+(150)(3)
$$

$$
=
-100+500+450
$$

$$
=
\boxed{850}.
$$

### Second sample

$$
\hat y_2
=
(-50)(1)+(100)(2)+(150)(2)
$$

$$
=
-50+200+300
$$

$$
=
\boxed{450}.
$$

### Third sample

$$
\hat y_3
=
(-50)(1)+(100)(2)+(150)(1)
$$

$$
=
-50+200+150
$$

$$
=
\boxed{300}.
$$

Thus,

$$
\boxed{
\mathbf w=
[-50,\;100,\;150]^T
}
$$

fits all three examples exactly.

In [ ]:
# Question 1 — batch gradient descent

X1 = np.array([
    [2.0, 5.0, 3.0],
    [1.0, 2.0, 2.0],
    [1.0, 2.0, 1.0]
])

y1 = np.array([850.0, 450.0, 300.0])

N = len(y1)
eta = 0.05
w = np.zeros(3)

loss_history = []

for epoch in range(100_000):
    predictions = X1 @ w
    error = predictions - y1

    loss = 0.5 * np.mean(error**2)
    gradient = (X1.T @ error) / N

    w = w - eta * gradient
    loss_history.append(loss)

    if np.linalg.norm(gradient) < 1e-10:
        break

print("Epochs =", epoch + 1)
print("Learned weights =", w)
print("Final predictions =", X1 @ w)
print("Targets =", y1)
print("Final loss =", 0.5*np.mean((X1 @ w - y1)**2))

## Why these weights are trustworthy

As an independent check, solve the exact linear system

$$
X\mathbf w=\mathbf y.
$$

Because $X$ is invertible,

$$
\mathbf w=X^{-1}\mathbf y.
$$

This gives exactly

$$
\boxed{
\mathbf w=
[-50,\;100,\;150]^T,
}
$$

which matches the gradient-descent result.

In [ ]:
# Independent exact/numerical check

w_direct = np.linalg.solve(X1, y1)

print("Direct linear-system solution =", w_direct)
print("Gradient-descent solution      =", w)
print("Difference                     =", w - w_direct)

In [ ]:
# Optional: visualize convergence of the loss.

plt.figure(figsize=(8,4.5))
plt.semilogy(loss_history)
plt.xlabel("Gradient-descent iteration")
plt.ylabel("Loss $J(w)$")
plt.title("Question 1 — Gradient Descent Convergence")
plt.grid(True, alpha=0.3)
plt.show()

# Question 2 — Two-Class Perceptron Classification

The professor gives:

### Blue class

$$
B_1=(-1,-1),
\qquad
B_2=(1,1),
\qquad
B_3=(2,2).
$$

### Red class

$$
R_1=(0,-1),
\qquad
R_2=(-1,0),
\qquad
R_3=(2,1),
\qquad
R_4=(1,2).
$$

Notice an important pattern.

For all blue points,

$$
x_1-x_2=0.
$$

For all red points,

$$
x_1-x_2=\pm1.
$$

So the blue points lie on the diagonal

$$
x_1=x_2,
$$

while the red points lie one unit to either side of that diagonal.

In [ ]:
blue = np.array([
    [-1.0, -1.0],
    [ 1.0,  1.0],
    [ 2.0,  2.0]
])

red = np.array([
    [ 0.0, -1.0],
    [-1.0,  0.0],
    [ 2.0,  1.0],
    [ 1.0,  2.0]
])

plt.figure(figsize=(7,6))
plt.scatter(blue[:,0], blue[:,1], s=90, marker="o", label="Blue")
plt.scatter(red[:,0], red[:,1], s=90, marker="x", label="Red")

xx = np.linspace(-1.5, 2.5, 300)
plt.plot(xx, xx, linestyle=":", label="$x_1=x_2$")

plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Question 2 — Professor's Two-Class Data")
plt.xlim(-1.5,2.5)
plt.ylim(-1.5,2.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Question 2(a) — Can One Perceptron Classify the Data?

A single perceptron creates one linear decision boundary:

$$
w_1x_1+w_2x_2+b=0.
$$

For perfect classification, the two classes must be linearly separable.

Here they are **not** linearly separable.

### A rigorous geometric proof

Consider the two red points

$$
R_1=(0,-1)
$$

and

$$
R_2=(-1,0).
$$

Their midpoint is

$$
\frac{R_1+R_2}{2}
=
\left(-\frac12,-\frac12\right).
$$

But

$$
\left(-\frac12,-\frac12\right)
$$

also lies on the blue line segment joining

$$
(-1,-1)
$$

to

$$
(1,1).
$$

Therefore the convex hull of the red class intersects the convex hull of the blue class.

Two classes whose convex hulls intersect cannot be strictly separated by one hyperplane.

Hence,

$$
\boxed{
\text{No, one perceptron cannot perfectly classify these data.}
}
$$

## Question 2(b) — Construct a Two-Perceptron Network by Hand

Since blue points satisfy

$$
x_1-x_2=0
$$

and red points satisfy

$$
|x_1-x_2|=1,
$$

we can surround the blue diagonal by a narrow strip.

Choose the two boundaries

$$
x_1-x_2=\frac12
$$

and

$$
x_2-x_1=\frac12.
$$

These are equivalent to

$$
x_1-x_2-\frac12=0
$$

and

$$
-x_1+x_2-\frac12=0.
$$

---

### Perceptron 1

Define

$$
a_1
=
x_1-x_2-\frac12.
$$

So

$$
\boxed{
\mathbf w^{(1)}
=
\begin{bmatrix}
1\\
-1
\end{bmatrix},
\qquad
b_1=-\frac12.
}
$$

Use the hard perceptron activation

$$
P(a)
=
\begin{cases}
1,&a\ge0,\\
0,&a<0.
\end{cases}
$$

This neuron activates for points on the side

$$
x_1-x_2\ge\frac12.
$$

---

### Perceptron 2

Define

$$
a_2
=
-x_1+x_2-\frac12.
$$

Therefore,

$$
\boxed{
\mathbf w^{(2)}
=
\begin{bmatrix}
-1\\
1
\end{bmatrix},
\qquad
b_2=-\frac12.
}
$$

This neuron activates for

$$
x_2-x_1\ge\frac12.
$$

---

## Network classification rule

The blue points lie **between** the two boundaries.

Therefore:

$$
\boxed{
\text{Blue if }(P(a_1),P(a_2))=(0,0).
}
$$

A red point is outside the strip on one side or the other:

$$
\boxed{
\text{Red if at least one perceptron output is }1.
}
$$

Equivalently,

$$
\text{Red if }P(a_1)\lor P(a_2)=1.
$$

---

## Verify every training point

| Point | Class | $a_1$ | $a_2$ | Outputs | Predicted |
|---|---|---:|---:|---|---|
| $(-1,-1)$ | Blue | $-0.5$ | $-0.5$ | $(0,0)$ | Blue |
| $(1,1)$ | Blue | $-0.5$ | $-0.5$ | $(0,0)$ | Blue |
| $(2,2)$ | Blue | $-0.5$ | $-0.5$ | $(0,0)$ | Blue |
| $(0,-1)$ | Red | $0.5$ | $-1.5$ | $(1,0)$ | Red |
| $(-1,0)$ | Red | $-1.5$ | $0.5$ | $(0,1)$ | Red |
| $(2,1)$ | Red | $0.5$ | $-1.5$ | $(1,0)$ | Red |
| $(1,2)$ | Red | $-1.5$ | $0.5$ | $(0,1)$ | Red |

Therefore the two-perceptron construction classifies all the professor's points correctly.

In [ ]:
# Question 2(b) — verify the two-perceptron network

def hard_perceptron(a):
    return 1 if a >= 0 else 0

wA = np.array([1.0, -1.0])
bA = -0.5

wB = np.array([-1.0, 1.0])
bB = -0.5

all_points = np.vstack([blue, red])
true_labels = np.array(["Blue"]*len(blue) + ["Red"]*len(red))

print("Point       a1      a2    outputs    predicted")
print("-"*58)

predicted_labels = []

for point, true_label in zip(all_points, true_labels):
    a1 = wA @ point + bA
    a2 = wB @ point + bB

    p1 = hard_perceptron(a1)
    p2 = hard_perceptron(a2)

    predicted = "Blue" if (p1 == 0 and p2 == 0) else "Red"
    predicted_labels.append(predicted)

    print(
        f"{tuple(point):12s} "
        if False else
        f"{str(tuple(point)):14s} {a1:6.2f} {a2:7.2f}   ({p1},{p2})      {predicted}"
    )

print("\nAll correct =", np.array_equal(np.array(predicted_labels), true_labels))

In [ ]:
# Visualize the two decision boundaries.

xx = np.linspace(-1.5, 2.5, 300)

# x1 - x2 = 0.5 -> x2 = x1 - 0.5
line1 = xx - 0.5

# x2 - x1 = 0.5 -> x2 = x1 + 0.5
line2 = xx + 0.5

plt.figure(figsize=(7,6))
plt.scatter(blue[:,0], blue[:,1], s=90, marker="o", label="Blue")
plt.scatter(red[:,0], red[:,1], s=90, marker="x", label="Red")

plt.plot(xx, line1, label="$x_2=x_1-0.5$")
plt.plot(xx, line2, label="$x_2=x_1+0.5$")

plt.fill_between(xx, line1, line2, alpha=0.15, label="Blue strip")

plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Question 2(b) — Two-Perceptron Classifier")
plt.xlim(-1.5,2.5)
plt.ylim(-1.5,2.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Question 3 — Python Code for Question 1

The professor asks for Python or MATLAB code implementing the gradient-descent solution from Question 1.

The following cell is a clean standalone implementation.

It uses:

$$
J(\mathbf w)
=
\frac{1}{2N}
\|X\mathbf w-\mathbf y\|^2
$$

and

$$
\nabla J
=
\frac1N
X^T(X\mathbf w-\mathbf y).
$$

The initialization and learning rate are explicitly stated so the experiment is reproducible.

In [ ]:
# Question 3 — standalone gradient-descent code

import numpy as np

X = np.array([
    [2.0, 5.0, 3.0],
    [1.0, 2.0, 2.0],
    [1.0, 2.0, 1.0]
])

y = np.array([850.0, 450.0, 300.0])

w = np.zeros(3)
eta = 0.05
N = len(y)

for epoch in range(100_000):

    y_hat = X @ w
    error = y_hat - y

    gradient = (X.T @ error) / N

    w = w - eta * gradient

    if np.linalg.norm(gradient) < 1e-10:
        break

print("Iterations:", epoch + 1)
print("Weights:", w)
print("Predictions:", X @ w)
print("Targets:", y)

The program returns approximately

$$
\boxed{
\mathbf w=
[-50,\;100,\;150]^T.
}
$$

# Question 4 — Supervised or Unsupervised Learning?

Question 1 is a **supervised learning** problem.

Why?

The training data contain both:

- input vectors $\mathbf{x}_i$;
- known desired outputs $y_i$.

For example,

$$
\mathbf{x}_1
=
[2,5,3]^T
$$

is paired with the target

$$
y_1=850.
$$

Gradient descent compares the neuron's prediction

$$
\hat y_i=\mathbf w^T\mathbf x_i
$$

with the known target

$$
y_i
$$

and uses the error to update the weights.

Therefore the desired answers supervise the learning process.

Hence,

$$
\boxed{
\text{Question 1 is supervised learning.}
}
$$

It is not unsupervised learning because the target outputs are explicitly supplied.

# Question 5 — Classify the Question 2 Data Without `sigmoid` or `sign(.)`

The professor asks us to classify the Question 2 data **without using the sigmoid or sign function**.

We can use the raw linear neuron outputs from the two-perceptron construction:

$$
a_1
=
x_1-x_2-\frac12
$$

and

$$
a_2
=
-x_1+x_2-\frac12.
$$

No sigmoid or `sign(.)` call is needed.

The classification can be read directly from the inequalities.

### Blue rule

A point is inside the diagonal strip when

$$
a_1<0
$$

and

$$
a_2<0.
$$

Therefore,

$$
\boxed{
\text{Blue if }a_1<0\text{ and }a_2<0.
}
$$

### Red rule

A red point lies outside the strip, so at least one raw score is nonnegative:

$$
\boxed{
\text{Red if }a_1\ge0\text{ or }a_2\ge0.
}
$$

---

## Raw-score table

| Point | True class | $a_1$ | $a_2$ | Classification |
|---|---|---:|---:|---|
| $(-1,-1)$ | Blue | $-0.5$ | $-0.5$ | Blue |
| $(1,1)$ | Blue | $-0.5$ | $-0.5$ | Blue |
| $(2,2)$ | Blue | $-0.5$ | $-0.5$ | Blue |
| $(0,-1)$ | Red | $0.5$ | $-1.5$ | Red |
| $(-1,0)$ | Red | $-1.5$ | $0.5$ | Red |
| $(2,1)$ | Red | $0.5$ | $-1.5$ | Red |
| $(1,2)$ | Red | $-1.5$ | $0.5$ | Red |

All seven points are classified correctly.

---

## Do we see a difference compared with the perceptron solution?

The **class labels are the same**.

The difference is in the information retained.

A hard perceptron converts the score immediately to a binary output such as 0 or 1.

The raw linear outputs retain their magnitudes:

$$
a_1,\;a_2.
$$

That magnitude tells us how far the point is from each decision boundary in score space.

For the true geometric perpendicular distance, divide the absolute score by

$$
\|\mathbf w\|.
$$

For example, for the first boundary,

$$
\text{distance}
=
\frac{|a_1|}
{\sqrt{1^2+(-1)^2}}
=
\frac{|a_1|}{\sqrt2}.
$$

Therefore:

$$
\boxed{
\text{same classifications, but raw scores retain margin information that a hard threshold discards.}
}
$$

In [ ]:
# Question 5 — raw-score classification without sigmoid or sign()

def classify_raw(point):
    a1 = wA @ point + bA
    a2 = wB @ point + bB

    if a1 < 0 and a2 < 0:
        return "Blue", a1, a2
    else:
        return "Red", a1, a2

for point, true_label in zip(all_points, true_labels):
    pred, a1, a2 = classify_raw(point)
    print(
        f"{str(tuple(point)):14s} "
        f"a1={a1:5.2f}  a2={a2:5.2f}  "
        f"pred={pred:4s}  true={true_label}"
    )

# Question 6 — Classify $[2,\;1.75]^T$ Using Perceptron Activation

The new input is

$$
\mathbf x
=
\begin{bmatrix}
2\\
1.75
\end{bmatrix}.
$$

Using the two perceptrons from Question 2:

$$
a_1
=
x_1-x_2-\frac12
$$

and

$$
a_2
=
-x_1+x_2-\frac12.
$$

Calculate the first score:

$$
a_1
=
2-1.75-0.5
=
-0.25.
$$

Calculate the second score:

$$
a_2
=
-2+1.75-0.5
=
-0.75.
$$

With the hard perceptron activation

$$
P(a)
=
\begin{cases}
1,&a\ge0,\\
0,&a<0,
\end{cases}
$$

we obtain

$$
P(a_1)=0
$$

and

$$
P(a_2)=0.
$$

The output code is therefore

$$
(0,0).
$$

From the network rule,

$$
(0,0)\Rightarrow\text{Blue}.
$$

Thus,

$$
\boxed{
[2,\;1.75]^T
\text{ is classified as Blue.}
}
$$

In [ ]:
# Question 6 verification

x_new = np.array([2.0, 1.75])

a1_new = wA @ x_new + bA
a2_new = wB @ x_new + bB

p1_new = hard_perceptron(a1_new)
p2_new = hard_perceptron(a2_new)

class_new = "Blue" if (p1_new == 0 and p2_new == 0) else "Red"

print("a1 =", a1_new)
print("a2 =", a2_new)
print("Perceptron outputs =", (p1_new, p2_new))
print("Class =", class_new)

# Question 7 — Use the Sigmoid Function and Discuss Probability

The sigmoid function is

$$
\boxed{
\sigma(a)
=
\frac{1}{1+e^{-a}}.
}
$$

For the same point

$$
\mathbf x=
[2,\;1.75]^T,
$$

we found

$$
a_1=-0.25
$$

and

$$
a_2=-0.75.
$$

---

## First sigmoid output

$$
\sigma(a_1)
=
\sigma(-0.25)
=
\frac{1}{1+e^{0.25}}.
$$

Therefore,

$$
\boxed{
\sigma(a_1)
\approx
0.4378234991.
}
$$

---

## Second sigmoid output

$$
\sigma(a_2)
=
\sigma(-0.75)
=
\frac{1}{1+e^{0.75}}.
$$

Hence,

$$
\boxed{
\sigma(a_2)
\approx
0.3208213008.
}
$$

Both values are below

$$
0.5.
$$

Using a $0.5$ threshold, both neurons remain inactive, so the point is again classified as

$$
\boxed{\text{Blue}.}
$$

---

## What is the estimated probability value?

With this two-perceptron construction, the sigmoid produces **two probability-like scores**, one for each red-side detector:

$$
\boxed{
p_1\approx0.4378
}
$$

and

$$
\boxed{
p_2\approx0.3208.
}
$$

The stronger red-side score is approximately

$$
\boxed{0.4378\text{ or }43.78\%}.
$$

Since neither detector reaches $0.5$, the hard class remains Blue.

### Important interpretation

These two sigmoid outputs are not automatically a single normalized probability distribution over the classes because the network contains two independent boundary detectors.

A true single class probability would require a defined probability-combination rule or a normalized output architecture such as softmax.

For this homework, the useful conclusion is:

$$
\boxed{
\text{sigmoid gives graded confidence values instead of only a hard 0/1 decision.}
}
$$

---

## Advantage of using probability-like values

A hard perceptron only tells us

$$
0
\quad\text{or}\quad
1.
$$

A sigmoid provides a continuous value between 0 and 1.

This can be useful because it provides:

1. **confidence information** — a value near 0.5 is uncertain, while a value near 0 or 1 is more decisive;
2. **threshold flexibility** — the decision threshold can be changed according to the application;
3. **risk-sensitive decisions** — high-risk applications can require higher confidence;
4. **ranking** — samples can be ordered by confidence;
5. **better downstream decision making** — probability-like outputs can be combined with costs or other evidence.

Therefore,

$$
\boxed{
\text{the sigmoid preserves confidence information that a hard perceptron discards.}
}
$$

In [ ]:
# Question 7 verification

def sigmoid(a):
    return 1/(1+np.exp(-a))

prob1 = sigmoid(a1_new)
prob2 = sigmoid(a2_new)

print("Sigmoid score 1 =", prob1)
print("Sigmoid score 2 =", prob2)
print("Largest red-side sigmoid score =", max(prob1, prob2))
print("Class with 0.5 threshold =", "Blue" if (prob1 < 0.5 and prob2 < 0.5) else "Red")

# Complete Homework #2 Final Answer Summary

## Question 1

Using batch gradient descent on

$$
J(\mathbf w)
=
\frac{1}{2N}
\|X\mathbf w-\mathbf y\|^2,
$$

the learned weights are

$$
\boxed{
\mathbf w=
[-50,\;100,\;150]^T.
}
$$

They reproduce the targets exactly:

$$
850,\qquad450,\qquad300.
$$

---

## Question 2(a)

A single perceptron cannot perfectly separate the two classes.

$$
\boxed{
\text{No single linear classifier exists.}
}
$$

---

## Question 2(b)

Use

$$
\boxed{
\mathbf w^{(1)}=[1,-1]^T,\qquad b_1=-0.5
}
$$

and

$$
\boxed{
\mathbf w^{(2)}=[-1,1]^T,\qquad b_2=-0.5.
}
$$

Blue:

$$
(P_1,P_2)=(0,0).
$$

Red:

$$
P_1=1
\quad\text{or}\quad
P_2=1.
$$

---

## Question 3

The Python gradient-descent implementation converges to

$$
\boxed{
[-50,\;100,\;150]^T.
}
$$

---

## Question 4

$$
\boxed{
\text{Question 1 is supervised learning.}
}
$$

---

## Question 5

Using the raw neuron scores,

$$
a_1=x_1-x_2-0.5,
$$

$$
a_2=-x_1+x_2-0.5,
$$

classify:

$$
\boxed{
\text{Blue if }a_1<0\text{ and }a_2<0
}
$$

and

$$
\boxed{
\text{Red otherwise}.
}
$$

The raw-score and hard-perceptron methods give the same training classifications, but the raw scores preserve margin information.

---

## Question 6

For

$$
[2,\;1.75]^T:
$$

$$
a_1=-0.25,
\qquad
a_2=-0.75.
$$

Hard perceptron outputs:

$$
(0,0).
$$

Therefore,

$$
\boxed{\text{Blue}.}
$$

---

## Question 7

Sigmoid scores:

$$
\boxed{
\sigma(-0.25)
\approx0.4378234991
}
$$

$$
\boxed{
\sigma(-0.75)
\approx0.3208213008.
}
$$

Both are below $0.5$, so the classification remains

$$
\boxed{\text{Blue}.}
$$

The larger red-side confidence score is approximately

$$
\boxed{43.78\%}.
$$

The advantage of sigmoid output is that it provides graded confidence rather than only a binary decision.

# MENG 417 Homework #2 — Mature Cheat Sheet

## 1. Linear neuron

A linear neuron computes

$$
\boxed{
\hat y
=
\mathbf w^T\mathbf x+b.
}
$$

If the problem gives only a weight vector and no bias, use

$$
\hat y=\mathbf w^T\mathbf x.
$$

---

## 2. Squared-error loss

For $N$ training examples,

$$
\boxed{
J(\mathbf w)
=
\frac{1}{2N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2.
}
$$

The factor $1/2$ simplifies the derivative.

---

## 3. Gradient of linear regression loss

For

$$
\hat{\mathbf y}=X\mathbf w,
$$

$$
\boxed{
\nabla J
=
\frac1N
X^T(X\mathbf w-\mathbf y).
}
$$

Gradient descent:

$$
\boxed{
\mathbf w
\leftarrow
\mathbf w-\eta\nabla J.
}
$$

---

## 4. Learning rate

The learning rate

$$
\eta
$$

controls the update size.

Too large:

$$
\text{training may diverge}.
$$

Too small:

$$
\text{training converges slowly}.
$$

Always state the learning rate when the professor has not specified one.

---

## 5. Supervised learning

Supervised learning uses input-target pairs:

$$
(\mathbf x_i,y_i).
$$

The model learns from the difference between prediction and known target.

---

## 6. Linear separability

One perceptron can only create one linear boundary:

$$
\boxed{
\mathbf w^T\mathbf x+b=0.
}
$$

If no single line/hyperplane separates the classes, one perceptron cannot solve the task perfectly.

---

## 7. Convex-hull test

If the convex hulls of the two classes intersect, the classes cannot be strictly linearly separated.

This gives a rigorous way to show why Question 2(a) has no single-perceptron solution.

---

## 8. Multiple perceptrons

Multiple perceptrons can define multiple boundaries.

In Question 2:

$$
x_1-x_2=0.5
$$

and

$$
x_2-x_1=0.5
$$

form a strip around

$$
x_1=x_2.
$$

The blue class is inside the strip; the red class is outside.

---

## 9. Hard perceptron activation

One common hard activation is

$$
P(a)
=
\begin{cases}
1,&a\ge0,\\
0,&a<0.
\end{cases}
$$

It removes the magnitude information and keeps only the class-side decision.

---

## 10. Raw neuron score

Before activation,

$$
a=\mathbf w^T\mathbf x+b.
$$

The sign identifies the side of the boundary.

The magnitude tells us how strongly the point lies on that side in score space.

Geometric perpendicular distance is

$$
\boxed{
\frac{|a|}{\|\mathbf w\|}.
}
$$

---

## 11. Sigmoid

$$
\boxed{
\sigma(a)
=
\frac{1}{1+e^{-a}}.
}
$$

Properties:

$$
0<\sigma(a)<1.
$$

$$
\sigma(0)=0.5.
$$

If

$$
a>0,
$$

then

$$
\sigma(a)>0.5.
$$

If

$$
a<0,
$$

then

$$
\sigma(a)<0.5.
$$

---

## 12. Hard output vs sigmoid output

Hard perceptron:

$$
\boxed{0\text{ or }1}
$$

Sigmoid:

$$
\boxed{\text{continuous value between }0\text{ and }1}
$$

Sigmoid therefore gives confidence-like information.

---

## 13. Probability caution

A sigmoid output can be interpreted probabilistically when the model and training objective support that interpretation.

With multiple independent sigmoid neurons, the outputs do **not** automatically sum to 1.

For mutually exclusive class probabilities, a softmax output is often used.

---

# Common Mistakes to Avoid

1. Do not add a bias to Question 1 unless you explicitly redefine the model; the professor asks for three weights matching three input components.
2. Do not omit the loss function when explaining gradient descent.
3. Do not claim Question 2 is linearly separable with one perceptron.
4. Do not use one boundary when the geometry requires a strip formed by two boundaries.
5. Do not confuse raw linear scores with activated outputs.
6. Do not call two independent sigmoid scores a normalized class-probability distribution.
7. Always state the threshold convention used by the perceptron.
8. Always verify learned weights by substituting them back into the training examples.

---

# Final Mental Model

Homework #2 connects three ideas:

$$
\boxed{
\text{learn weights with gradient descent}
}
$$

$$
\boxed{
\text{use multiple linear boundaries when one is insufficient}
}
$$

$$
\boxed{
\text{replace hard decisions with sigmoid scores when confidence is useful}.
}
$$

# Final Numerical Verification

The following checks independently verify the key fixed answers.

In [ ]:
# Q1 exact solution
assert np.allclose(np.linalg.solve(X1, y1), np.array([-50.,100.,150.]), atol=1e-10)

# Q1 predictions
assert np.allclose(X1 @ np.array([-50.,100.,150.]), y1)

# Q2 two-perceptron classifier
expected_labels = np.array(["Blue","Blue","Blue","Red","Red","Red","Red"])

actual_labels = []
for point in all_points:
    a1 = wA @ point + bA
    a2 = wB @ point + bB
    actual_labels.append("Blue" if (a1 < 0 and a2 < 0) else "Red")

assert np.array_equal(np.array(actual_labels), expected_labels)

# Q6
assert abs(a1_new - (-0.25)) < 1e-12
assert abs(a2_new - (-0.75)) < 1e-12
assert class_new == "Blue"

# Q7
assert abs(prob1 - 0.43782349911420193) < 1e-12
assert abs(prob2 - 0.320821300824607) < 1e-12

print("All Homework #2 fixed mathematical checks passed.")